# Gemma 4 E2B — Live Weights Test

Validates the TransformerLens Gemma 4 adapter against actual model weights.
Checks: weight loading, forward pass, dual d_head, PLE flow, shared KV correctness.

**Assumes**: weights are available locally at `MODEL_PATH`; no HF token required.
**Install**: pip installs from our fork's `gemma4-support` branch.

In [ ]:
import subprocess, sys

# --no-deps: only reinstall TL itself, don't touch torch/transformers/etc.
# Without this, --force-reinstall pulls a fresh torch build and kills your CUDA setup.
for cmd in [
    [sys.executable, "-m", "pip", "uninstall", "-y", "transformer_lens"],
    [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", "--no-cache-dir", "-q",
     "git+https://github.com/hebenon/Gemma4-TransformerLens.git@gemma4-support"],
]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print("OK" if r.returncode == 0 else "FAIL: " + r.stderr[-300:])

# Verify the right file is loaded
import transformer_lens
print("Loaded from:", transformer_lens.__file__)
print("\n*** RESTART KERNEL NOW, then run from cell 2 ***")

In [ ]:
import importlib.metadata
import torch
import transformer_lens
from transformer_lens import HookedTransformer

tl_version = importlib.metadata.version("transformer_lens")
print("TransformerLens:", tl_version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "/", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
# Path to the HF checkpoint directory (adjust to Kathleen mount point)
MODEL_PATH = "/home/ubuntu/models/gemma-4-E2B-it"   # <-- edit this

DTYPE = torch.bfloat16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}, dtype: {DTYPE}")

## 1. Load model

In [ ]:
from transformers import AutoModel
import gc, ctypes, transformer_lens.loading_from_pretrained as loading

WEIGHTS_PATH = "/kaggle/working/tl_gemma4_weights.pt"

def _trim_ram():
    """gc.collect() frees Python objects; malloc_trim(0) returns freed heap pages to OS."""
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)

print("Stage 1: loading HF model on CPU...")
hf_model = AutoModel.from_pretrained(
    MODEL_PATH, torch_dtype=DTYPE, trust_remote_code=True, device_map="cpu",
    low_cpu_mem_usage=True,
)
print(f"  HF model type: {type(hf_model).__name__}")

print("Stage 2: building TL state dict on CPU...")
cfg = loading.get_pretrained_model_config(
    "google/gemma-4-E2B-it", fold_ln=False, dtype=DTYPE,
)
state_dict = loading.get_pretrained_state_dict(
    "google/gemma-4-E2B-it", cfg, hf_model, dtype=DTYPE,
)
del hf_model; _trim_ram()
print(f"  State dict keys: {len(state_dict)}")

print(f"Stage 2b: saving state dict to disk ({WEIGHTS_PATH})...")
torch.save(state_dict, WEIGHTS_PATH)
del state_dict; _trim_ram()
print("  Freed state dict from RAM")

print("Stage 3: constructing empty TL model...")
# TL creates nn.Parameters in float32 by default, then casts — peak is
# ~17GB (f32) + ~8.65GB (bf16 copy) = ~26GB. Fix: create in bf16 directly.
from transformer_lens import HookedTransformer
_old_dtype = torch.get_default_dtype()
torch.set_default_dtype(DTYPE)
model = HookedTransformer(cfg, move_to_device=False)
torch.set_default_dtype(_old_dtype)

print("Stage 3b: moving model to GPU...")
model = model.to(DEVICE)
_trim_ram()
torch.cuda.empty_cache()

print("Stage 4: loading weights from disk (mmap → GPU params)...")
state_dict = torch.load(WEIGHTS_PATH, mmap=True, weights_only=True)
model.load_state_dict(state_dict, strict=False)
del state_dict; _trim_ram()
torch.cuda.empty_cache()

model.eval()
print("Done — param count:", sum(p.numel() for p in model.parameters()) / 1e9, "B")

## 2. Config sanity

In [ ]:
cfg = model.cfg

checks = [
    ("d_model",              cfg.d_model,             1536),
    ("n_layers",             cfg.n_layers,             35),
    ("n_heads",              cfg.n_heads,              8),
    ("d_head (local)",       cfg.d_head,               256),
    ("d_head_global",        cfg.d_head_global,        512),
    ("n_kv_heads",           cfg.n_key_value_heads,    1),
    ("d_mlp",                cfg.d_mlp,                6144),
    ("d_ple",                cfg.d_ple,                256),
    ("num_kv_shared_layers", cfg.num_kv_shared_layers, 20),
    ("rotary_dim_global",    cfg.rotary_dim_global,    128),
    ("window_size",          cfg.window_size,          512),
]

all_ok = True
for name, got, expected in checks:
    ok = got == expected
    all_ok = all_ok and ok
    print(f"  {'OK' if ok else 'FAIL'}  {name}: {got}" + (f" (expected {expected})" if not ok else ""))

print("\nConfig:", "PASS" if all_ok else "FAIL")

## 3. Forward pass — smoke test

In [ ]:
# Tokenize a short prompt
prompt = "The capital of France is"
tokens = model.to_tokens(prompt)   # [1, seq_len]
print("Tokens:", tokens.shape, tokens)

with torch.no_grad():
    logits = model(tokens)         # [1, seq_len, d_vocab]

print("Logits shape:", logits.shape)
print("Finite:", torch.isfinite(logits).all().item())

# Top-5 next token predictions
next_tok_logits = logits[0, -1]    # [d_vocab]
top5 = torch.topk(next_tok_logits, 5)
print("\nTop-5 continuations:")
for tok_id, score in zip(top5.indices, top5.values):
    print(f"  {score:.2f}  {repr(model.to_string(tok_id.unsqueeze(0)))}")

## 4. Dual d_head — verify per-layer weight shapes

In [ ]:
GLOBAL_LAYERS = {4, 9, 14, 19, 24, 29, 34}

head_dim_ok = True
for i, block in enumerate(model.blocks):
    expected_d_head = 512 if i in GLOBAL_LAYERS else 256
    # W_Q shape: [n_heads, d_model, d_head]
    actual_d_head = block.attn.W_Q.shape[-1]
    ok = actual_d_head == expected_d_head
    head_dim_ok = head_dim_ok and ok
    if not ok:
        print(f"  FAIL  block {i}: W_Q d_head={actual_d_head}, expected {expected_d_head}")

print("Dual d_head shapes:", "PASS" if head_dim_ok else "FAIL")

## 5. PLE — verify components load and flow

In [ ]:
# Check PLEPrecomputer exists and has correct weight shapes
assert hasattr(model, "ple"), "model.ple missing"
ple = model.ple

n_flat = cfg.n_layers * cfg.d_ple
checks_ple = [
    ("W_embed", ple.W_embed.shape, (cfg.ple_vocab_size, n_flat)),
    ("W_proj",  ple.W_proj.shape,  (cfg.d_model, n_flat)),
]
for name, got, expected in checks_ple:
    ok = got == torch.Size(expected)
    print(f"  {'OK' if ok else 'FAIL'}  ple.{name}: {got}" + (f" (expected {expected})" if not ok else ""))

# Check per-block PLE weights
block_ple_ok = True
for i, block in enumerate(model.blocks):
    ok = (
        hasattr(block, "ple_gate") and
        block.ple_gate.W.shape == torch.Size([cfg.d_model, cfg.d_ple]) and
        block.ple_up.W.shape  == torch.Size([cfg.d_ple, cfg.d_model])
    )
    if not ok:
        print(f"  FAIL  block {i} PLE weights wrong shape")
        block_ple_ok = False

print("PLE weight shapes:", "PASS" if block_ple_ok else "FAIL")

In [ ]:
# Verify PLE vector actually flows (hook_ple_input fires with correct shape)
ple_shapes_seen = {}

def _capture_ple(value, hook):
    layer = int(hook.name.split(".")[1])   # "blocks.N.hook_ple_input"
    ple_shapes_seen[layer] = value.shape
    return value

with torch.no_grad():
    model.run_with_hooks(
        tokens,
        fwd_hooks=[(f"blocks.{i}.hook_ple_input", _capture_ple) for i in range(cfg.n_layers)],
    )

B, L = tokens.shape
expected_ple_shape = torch.Size([B, L, cfg.d_ple])
all_ple_ok = all(s == expected_ple_shape for s in ple_shapes_seen.values())
print(f"PLE vectors fired in {len(ple_shapes_seen)}/{cfg.n_layers} blocks")
print(f"PLE shape per block: {next(iter(ple_shapes_seen.values()))} — {'PASS' if all_ple_ok else 'FAIL'}")

## 6. Shared KV — verify shared layers use same K/V as source

In [ ]:
# Capture K from source layers (13, 14) and from a sample of borrower layers.
# K from borrowers should be identical to the corresponding source K.
# (V likewise — we check layer 13 local source vs a local borrower, and 14 global.)

k_captures = {}
v_captures = {}

SAMPLE_BORROWERS = [15, 20, 25, 30]   # mix of local and global
LAYERS_TO_HOOK   = {13, 14} | set(SAMPLE_BORROWERS)

def _capture_k(value, hook, layer=None):
    k_captures[layer] = value.detach().cpu()
    return value

def _capture_v(value, hook, layer=None):
    v_captures[layer] = value.detach().cpu()
    return value

hooks = []
for i in LAYERS_TO_HOOK:
    hooks.append((f"blocks.{i}.attn.hook_k", lambda v, h, l=i: (_capture_k(v, h, l), v)[1]))
    hooks.append((f"blocks.{i}.attn.hook_v", lambda v, h, l=i: (_capture_v(v, h, l), v)[1]))

with torch.no_grad():
    model.run_with_hooks(tokens, fwd_hooks=hooks)

print("Shared KV verification (comparing borrower hook_k to source hook_k)")
print("Note: hook_k fires BEFORE k_norm — shared layers receive pre-norm K from HookedTransformer.")
print("The correctness guarantee is in _apply_ple / compute_kv_for_sharing (post-norm).")
print()

sources = cfg.kv_shared_layer_sources
for borrower in SAMPLE_BORROWERS:
    src = sources[borrower]
    attn_type = cfg.attn_types[borrower]
    k_src = k_captures[src]
    k_borrow = k_captures[borrower]
    # Shapes may differ (local=256, global=512 d_head) — only compare if same type
    if k_src.shape == k_borrow.shape:
        match = torch.allclose(k_src, k_borrow, atol=1e-3)
        print(f"  block {borrower:2d} ({attn_type:6s}) borrows from {src}: K match = {match}")
    else:
        print(f"  block {borrower:2d} ({attn_type:6s}) vs src {src}: shape mismatch {k_borrow.shape} vs {k_src.shape} (different attn types)")

## 7. Local vs global attention — head dimension in activations

In [ ]:
# Check Q/K/V activation shapes for a local and global block
q_shapes = {}

def _cap_q(value, hook, layer=None):
    q_shapes[layer] = value.shape
    return value

sample_layers = [0, 4]   # 0=local, 4=global
hooks = [(f"blocks.{i}.attn.hook_q", lambda v, h, l=i: (_cap_q(v, h, l), v)[1]) for i in sample_layers]

with torch.no_grad():
    model.run_with_hooks(tokens, fwd_hooks=hooks)

B, L = tokens.shape
for layer in sample_layers:
    shape = q_shapes[layer]
    attn_type = cfg.attn_types[layer]
    expected_d_head = 512 if attn_type == "global" else 256
    ok = shape == torch.Size([B, L, cfg.n_heads, expected_d_head])
    print(f"  {'OK' if ok else 'FAIL'}  block {layer:2d} ({attn_type:6s}): hook_q shape {shape} (expected d_head={expected_d_head})")

print("Q activation shapes:", "PASS" if all(
    q_shapes[l] == torch.Size([B, L, cfg.n_heads, 512 if cfg.attn_types[l] == 'global' else 256])
    for l in sample_layers
) else "FAIL")

## 8. Layer scale values — spot check

In [ ]:
# layer_scale should be loaded from HF layer_scalar weights.
# Values from architecture enumeration: range roughly 0.018–0.87.
# If all 1.0: weights didn't load (default init).
scales = [block.layer_scale.item() for block in model.blocks]
all_ones = all(abs(s - 1.0) < 1e-6 for s in scales)
print(f"layer_scale — first 5: {[f'{s:.4f}' for s in scales[:5]]}")
print(f"layer_scale — last  5: {[f'{s:.4f}' for s in scales[-5:]]}")
print(f"All 1.0 (not loaded): {all_ones}")
if not all_ones:
    print(f"Min: {min(scales):.4f}  Max: {max(scales):.4f}  — expected range ~0.01–0.9")

## 9. Generate a short sequence

In [ ]:
# Quick greedy generation — sanity check that the full pipeline works
from transformer_lens.utilities import generate

with torch.no_grad():
    out = model.generate(
        prompt,
        max_new_tokens=20,
        temperature=0,
        verbose=False,
    )

print("Generated:", repr(out))

## 10. Summary

In [ ]:
print("="*60)
print("Run all cells and check for FAIL markers above.")
print("Key things to flag:")
print("  - Any FAIL in config / d_head / PLE shape checks")
print("  - Non-finite logits")
print("  - PLE vectors not firing in all blocks")
print("  - layer_scale all 1.0 (means weight conversion missed it)")
print("  - Generated text should be coherent (Paris, etc.)")
print("="*60)